# Week 2, Notebook 2: Going Deeper — The Engineering of Deep Learning
## Initialization, Normalization, Loss Landscapes & Overparameterization

**What you'll build:** Deep networks that ACTUALLY train, plus loss landscape visualizations.

**Curriculum points:**
- ③ Overparameterization helps (and why)
- ⑥ Geometry matters — flat vs sharp minima
- ⑦ Capacity ≠ performance — He init, BatchNorm, Dropout

**Time estimate:** 60 minutes (the meatiest notebook)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from copy import deepcopy

torch.manual_seed(42)
np.random.seed(42)

# Generate data
def make_moons(n=800, noise=0.15):
    t = np.linspace(0, np.pi, n // 2)
    x1 = np.c_[np.cos(t), np.sin(t)] + np.random.randn(n // 2, 2) * noise
    x2 = np.c_[np.cos(t) + 0.5, -np.sin(t) + 0.5] + np.random.randn(n // 2, 2) * noise
    X = np.vstack([x1, x2])
    y = np.hstack([np.zeros(n // 2), np.ones(n // 2)])
    idx = np.random.permutation(n)
    X = (X[idx] - X[idx].mean(0)) / (X[idx].std(0) + 1e-8)
    return torch.FloatTensor(X), torch.FloatTensor(y[idx]).unsqueeze(1)

X_all, y_all = make_moons(800)
X_train, y_train = X_all[:640], y_all[:640]
X_val, y_val = X_all[640:], y_all[640:]
print(f"Data ready: {X_train.shape[0]} train, {X_val.shape[0]} val")

## Part 1: Initialization Matters — He vs Xavier vs Random

**Curriculum Point ⑦:** Architecture, initialization, normalization, and data quality dominate.

In [ ]:
# ============================================================
# EXPERIMENT: initialization strategies
# ============================================================

def make_deep_net(depth, width, init='he'):
    """Build a deep network with specified initialization."""
    layers = []
    in_dim = 2
    for i in range(depth):
        layer = nn.Linear(in_dim, width)
        
        # Apply initialization
        if init == 'he':
            nn.init.kaiming_normal_(layer.weight, mode='fan_in', nonlinearity='relu')
        elif init == 'xavier':
            nn.init.xavier_normal_(layer.weight)
        elif init == 'large_random':
            nn.init.normal_(layer.weight, std=1.0)  # Too large!
        elif init == 'tiny_random':
            nn.init.normal_(layer.weight, std=0.001)  # Too small!
        nn.init.zeros_(layer.bias)
        
        layers.append(layer)
        layers.append(nn.ReLU())
        in_dim = width
    
    # Output layer
    out_layer = nn.Linear(width, 1)
    nn.init.xavier_normal_(out_layer.weight)
    layers.append(out_layer)
    layers.append(nn.Sigmoid())
    
    return nn.Sequential(*layers)


def quick_train(model, epochs=1000, lr=0.01):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    losses = []
    for epoch in range(epochs):
        pred = model(X_train)
        loss = criterion(pred, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses


# Compare initializations on a 6-layer network
depth, width = 6, 32
inits = ['he', 'xavier', 'large_random', 'tiny_random']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for init_name in inits:
    torch.manual_seed(42)
    model = make_deep_net(depth, width, init=init_name)
    losses = quick_train(model, epochs=1500)
    
    n_params = sum(p.numel() for p in model.parameters())
    axes[0].plot(losses, label=f'{init_name} ({n_params} params)', alpha=0.8)

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title(f'Initialization Effect on {depth}-Layer Network')
axes[0].set_yscale('log')
axes[0].legend()
axes[0].grid(True, alpha=0.2)

# Show activation distributions for different inits
x_probe = X_train[:100]
for idx, init_name in enumerate(['he', 'large_random']):
    torch.manual_seed(42)
    model = make_deep_net(depth, width, init=init_name)
    
    # Record activations at each layer
    activations = []
    x = x_probe
    with torch.no_grad():
        for layer in model:
            x = layer(x)
            if isinstance(layer, nn.ReLU):
                activations.append(x.numpy().flatten())
    
    for layer_idx, act in enumerate(activations[:4]):
        axes[1].hist(act, bins=50, alpha=0.3, density=True,
                     label=f'{init_name} L{layer_idx}' if idx == 0 else None)

axes[1].set_title('Activation Distributions (He vs Large Random)')
axes[1].set_xlabel('Activation Value')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.2)

plt.suptitle('INITIALIZATION: He/Xavier vs Bad Init', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('w2_02_initialization.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n⚡ KEY: He init keeps activations well-scaled across layers.")
print("   Large random → exploding activations. Tiny → dead neurons.")

## Part 2: BatchNorm + Dropout — Making Depth Work

BatchNorm normalizes activations between layers → stabilizes training.  
Dropout randomly zeroes neurons → prevents co-adaptation → reduces overfitting.

In [ ]:
# ============================================================
# EXPERIMENT: Plain deep net vs BatchNorm vs BatchNorm+Dropout
# ============================================================

class DeepNetBN(nn.Module):
    """Deep network WITH BatchNorm and optional Dropout."""
    def __init__(self, depth=6, width=32, dropout=0.0):
        super().__init__()
        layers = []
        in_dim = 2
        for _ in range(depth):
            layers.append(nn.Linear(in_dim, width))
            layers.append(nn.BatchNorm1d(width))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_dim = width
        layers.append(nn.Linear(width, 1))
        layers.append(nn.Sigmoid())
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)


configs = {
    'Plain (He init)': lambda: make_deep_net(8, 32, 'he'),
    'BatchNorm': lambda: DeepNetBN(8, 32, dropout=0.0),
    'BatchNorm + Dropout(0.3)': lambda: DeepNetBN(8, 32, dropout=0.3),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#e74c3c', '#3498db', '#2ecc71']

for (name, model_fn), color in zip(configs.items(), colors):
    torch.manual_seed(42)
    model = model_fn()
    optimizer = optim.Adam(model.parameters(), lr=0.005)
    criterion = nn.MSELoss()
    
    train_losses, val_losses = [], []
    for epoch in range(2000):
        model.train()
        pred = model(X_train)
        loss = criterion(pred, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
        
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val), y_val)
            val_losses.append(val_loss.item())
    
    axes[0].plot(train_losses, color=color, alpha=0.8, label=f'{name} (train)')
    axes[1].plot(val_losses, color=color, alpha=0.8, label=f'{name} (val)')

for ax, title in zip(axes, ['Training Loss', 'Validation Loss']):
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title(title)
    ax.set_yscale('log')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)

plt.suptitle('8-LAYER NETWORK: Plain vs BatchNorm vs BN+Dropout', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('w2_02_batchnorm.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n⚡ BatchNorm stabilizes training of deep networks.")
print("   Dropout closes the train-val gap (reduces overfitting).")

## Part 3: Overparameterization — Why More Parameters Can Help

**Curriculum Point ③:** Modern networks often have MORE parameters than data points, yet generalize well.

In [ ]:
# ============================================================
# EXPERIMENT: Overparameterization — more params, better generalization?
# ============================================================
param_counts = []
val_accuracies = []
train_accuracies = []

widths = [4, 8, 16, 32, 64, 128, 256]

print(f"Dataset size: {X_train.shape[0]} training samples")
print(f"{'Width':>6} | {'Params':>8} | {'Ratio':>10} | {'Train Acc':>10} | {'Val Acc':>10}")
print("-" * 60)

for w in widths:
    torch.manual_seed(42)
    model = DeepNetBN(depth=3, width=w, dropout=0.0)
    optimizer = optim.Adam(model.parameters(), lr=0.005)
    criterion = nn.MSELoss()
    
    for epoch in range(2000):
        model.train()
        loss = criterion(model(X_train), y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    model.eval()
    with torch.no_grad():
        train_acc = ((model(X_train) > 0.5).float() == y_train).float().mean().item()
        val_acc = ((model(X_val) > 0.5).float() == y_val).float().mean().item()
    
    n_params = sum(p.numel() for p in model.parameters())
    ratio = n_params / X_train.shape[0]
    
    param_counts.append(n_params)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)
    
    marker = " ← overparameterized!" if ratio > 1 else ""
    print(f"{w:>6} | {n_params:>8} | {ratio:>9.2f}x | {train_acc:>9.3f} | {val_acc:>9.3f}{marker}")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(param_counts, train_accuracies, 'o-', color='steelblue', label='Train Acc', markersize=8)
ax.plot(param_counts, val_accuracies, 's-', color='coral', label='Val Acc', markersize=8)
ax.axvline(x=X_train.shape[0], color='gray', linestyle='--', alpha=0.5, label=f'N_train={X_train.shape[0]}')
ax.set_xlabel('Number of Parameters')
ax.set_ylabel('Accuracy')
ax.set_title('OVERPARAMETERIZATION: More Params Can Generalize Better')
ax.set_xscale('log')
ax.legend()
ax.grid(True, alpha=0.2)
ax.set_ylim(0.7, 1.02)
plt.tight_layout()
plt.savefig('w2_02_overparameterization.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n⚡ COUNTERINTUITIVE: Models with MORE params than data points can generalize BETTER.")
print("   Gradient descent + data structure = implicit regularization.")

## Part 4: Loss Landscape Visualization

**Curriculum Point ⑥:** Training is navigating a high-dimensional loss landscape.  
Flat minima → better generalization. Sharp minima → poor robustness.

In [ ]:
# ============================================================
# Loss landscape visualization (2D slice)
# ============================================================
# Method: take trained weights, pick 2 random directions, 
# evaluate loss along those directions

def get_params_vector(model):
    return torch.cat([p.data.flatten() for p in model.parameters()])

def set_params_vector(model, vec):
    offset = 0
    for p in model.parameters():
        n = p.numel()
        p.data = vec[offset:offset+n].reshape(p.shape)
        offset += n

def compute_loss_landscape(model, X, y, resolution=30, scale=1.0):
    """Compute loss on a 2D grid around current parameters."""
    criterion = nn.MSELoss()
    
    # Get current params and two random directions
    theta = get_params_vector(model).clone()
    d1 = torch.randn_like(theta)
    d2 = torch.randn_like(theta)
    
    # Normalize directions to have same norm as params
    d1 = d1 / d1.norm() * theta.norm()
    d2 = d2 / d2.norm() * theta.norm()
    
    alphas = np.linspace(-scale, scale, resolution)
    betas = np.linspace(-scale, scale, resolution)
    
    landscape = np.zeros((resolution, resolution))
    
    model.eval()
    for i, a in enumerate(alphas):
        for j, b in enumerate(betas):
            new_params = theta + a * d1 + b * d2
            set_params_vector(model, new_params)
            with torch.no_grad():
                loss = criterion(model(X), y)
            landscape[j, i] = loss.item()
    
    # Restore original params
    set_params_vector(model, theta)
    
    return alphas, betas, landscape


# Train two models: one with BatchNorm (flat minimum), one without (sharp minimum)
torch.manual_seed(42)
model_flat = DeepNetBN(depth=4, width=32, dropout=0.1)
optimizer = optim.Adam(model_flat.parameters(), lr=0.005)
criterion = nn.MSELoss()
for _ in range(2000):
    model_flat.train()
    loss = criterion(model_flat(X_train), y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

torch.manual_seed(42)
model_sharp = make_deep_net(4, 32, 'he')
optimizer = optim.SGD(model_sharp.parameters(), lr=0.1)
for _ in range(2000):
    loss = criterion(model_sharp(X_train), y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Compute landscapes
print("Computing loss landscapes (this may take a moment)...")
a1, b1, L1 = compute_loss_landscape(model_flat, X_val, y_val, resolution=25, scale=0.3)
a2, b2, L2 = compute_loss_landscape(model_sharp, X_val, y_val, resolution=25, scale=0.3)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, alphas, betas, landscape, title in [
    (axes[0], a1, b1, L1, 'BN+Dropout (flatter minimum)'),
    (axes[1], a2, b2, L2, 'Plain SGD (sharper minimum)')
]:
    AA, BB = np.meshgrid(alphas, betas)
    # Clip for visualization
    landscape_clipped = np.clip(landscape, 0, np.percentile(landscape, 95))
    c = ax.contourf(AA, BB, landscape_clipped, levels=30, cmap='viridis')
    ax.contour(AA, BB, landscape_clipped, levels=15, colors='white', linewidths=0.3, alpha=0.5)
    ax.plot(0, 0, 'r*', markersize=15, label='Trained weights')
    ax.set_xlabel('Direction 1')
    ax.set_ylabel('Direction 2')
    ax.set_title(title)
    ax.legend()
    plt.colorbar(c, ax=ax, label='Loss')

plt.suptitle('LOSS LANDSCAPE: Flat Minima ↔ Better Generalization', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('w2_02_loss_landscape.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n⚡ Flat minima (left) generalize better than sharp minima (right).")
print("   BatchNorm + proper training → flatter loss landscape.")

## ✅ Week 2 Complete — Self-Assessment

### You should now be able to:
- [ ] Choose He init for ReLU, Xavier for sigmoid/tanh — and explain WHY
- [ ] Add BatchNorm + Dropout to any PyTorch network and predict the effect
- [ ] Explain the overparameterization paradox in simple terms
- [ ] Visualize and interpret loss landscapes
- [ ] Debug a model that isn't training (check: init? LR? architecture? data?)

### The "Karpathy Checklist" (internalize this):
1. ✅ Start simple (small model, small data, overfit first)
2. ✅ Add complexity gradually
3. ✅ Always compare train vs val
4. ✅ Visualize everything
5. ✅ Change ONE thing at a time

## ➡️ Week 3: Generative AI!
Open `W3_01_Pure_Autoencoder.ipynb`

## Visualizing a Deep Architecture

A conceptual diagram of the deeper network, hinting at the need for skip connections/residuals when going very deep.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.DiGraph()
# Abstract representation for deep blocks
G.add_edge("Input", "Layer 1")
G.add_edge("Layer 1", "ReLU 1")
G.add_edge("ReLU 1", "Layer 2")
G.add_edge("Layer 2", "ReLU 2")
G.add_edge("ReLU 2", "Layer 3")
G.add_edge("Layer 3", "Output")
# Skip connection to illustrate Going Deeper strategies
G.add_edge("Input", "Layer 3", style="dashed", label="Residual?")

pos = {"Input": (0, 0), "Layer 1": (1, 0), "ReLU 1": (2, 0), "Layer 2": (3, 0), "ReLU 2": (4, 0), "Layer 3": (5, 0), "Output": (6, 0)}
plt.figure(figsize=(12, 3))
nx.draw(G, pos, with_labels=True, node_color='orange', node_size=3000, node_shape="s")
nx.draw_networkx_edges(G, pos, edgelist=[("Input", "Layer 3")], style="dashed", connectionstyle="arc3,rad=-0.5")
plt.title("Deep Network Architecture Overview")

plt.savefig('w2_02_network_arch.png', dpi=100, bbox_inches='tight')
plt.show()
